In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (HLPpred-Fuse)

This notebook curates the **HLPpred-Fuse** dataset from FASTA files. Peptide sequences are parsed from multiple inputs and binary labels are inferred directly from FASTA record identifiers (headers). The pipeline then applies duplicate consistency checks, builds metadata, and exports a standardized dataset for downstream analysis.

- **Toxic effect / endpoint:** hemolytic
- **Source:** HLPpred-Fuse
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads FASTA-like files** (`.fasta`, `.fa`, `.faa`, `.txt`) from the HLPpred-Fuse input directory and concatenates all records.
- **Infers labels from FASTA record IDs** using simple string matching:
  - if `id` contains `"Positive"` (case-insensitive) → `label = 1`,
  - if `id` contains `"Negative"` (case-insensitive) → `label = 0`,
  - otherwise `label = NA` (missing/unknown).
- **Keeps a standardized schema**:
  - `sequence`
  - `label`
- **Checks duplicated sequences** by sequence:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv`,
  - `detected_error_sequences.csv`,
  - `metadata.json`.

In [2]:
name_source = "HLPpred-Fuse"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants.
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
input_dir = Path(PATH_INPUT) / name_source
valid_ext = {".fasta", ".fa", ".faa", ".txt"}
dfs = []

for file in input_dir.iterdir():
    if file.is_file() and file.suffix.lower() in valid_ext:
        df = read_fasta_doc(file)
        dfs.append(df)

df_HLPpred_Fuse = pd.concat(dfs, ignore_index=True)

In [4]:
df_HLPpred_Fuse["label"] = pd.NA 

df_HLPpred_Fuse.loc[df_HLPpred_Fuse["id"].str.contains("Positive", case=False, na=False), "label"] = 1
df_HLPpred_Fuse.loc[df_HLPpred_Fuse["id"].str.contains("Negative", case=False, na=False), "label"] = 0

df_HLPpred_Fuse = df_HLPpred_Fuse[["sequence", "label"]]

- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_HLPpred_Fuse, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_full.shape

(4365, 2)

In [7]:
df_errors.shape

(123, 1)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_HLPpred_Fuse)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2020, 4, 20, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from another DB',
 'repository or server': 'http://thegleelab.org/HLPpred-Fuse/FuseData.html',
 'publication': 'https://academic.oup.com/bioinformatics/article/36/11/3350/5799076?login=false',
 'number_of_raw_sequences': 5063,
 'number_of_sequences_retained': 4365,
 'number_of_positive_sequences': 1360,
 'number_of_negative_sequences': 3005,
 'number_of_erroneous_sequences': 123,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)